In [21]:
#Import Libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score
)

import joblib

In [22]:
#Dataset loading
botnet=pd.read_csv("../CIC-IDS2017-Dataset/Botnet-Friday-no-metadata.parquet.csv")
bruteforce=pd.read_csv("../CIC-IDS2017-Dataset/Bruteforce-Tuesday-no-metadata.parquet.csv")
ddos=pd.read_csv("../CIC-IDS2017-Dataset/DDoS-Friday-no-metadata.parquet.csv")
dos=pd.read_csv("../CIC-IDS2017-Dataset/DoS-Wednesday-no-metadata.parquet.csv")
infiltration=pd.read_csv("../CIC-IDS2017-Dataset/Infiltration-Thursday-no-metadata.parquet.csv")
portscan=pd.read_csv("../CIC-IDS2017-Dataset/Portscan-Friday-no-metadata.parquet.csv")
webattack=pd.read_csv("../CIC-IDS2017-Dataset/WebAttacks-Thursday-no-metadata.parquet.csv")

In [23]:
#Creating Master Dataset
master_df = pd.concat(
    [
        botnet,
        bruteforce,
        ddos,
        dos,
        infiltration,
        portscan,
        webattack
    ],
    ignore_index=True
)

print(master_df.shape)
master_df.head()

(1854979, 78)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,112740690,32,16,6448,1152,403,0,201.5,204.7242,...,32,3.594286e+02,1.199802e+01,380,343,16100000.0,498804.80,16400000,15400000,Benign
1,6,112740560,32,16,6448,5056,403,0,201.5,204.7242,...,32,3.202857e+02,1.574499e+01,330,285,16100000.0,498793.66,16400000,15400000,Benign
2,0,113757377,545,0,0,0,0,0,0.0,0.0000,...,0,9.361829e+06,7.324646e+06,18900000,19,12200000.0,6935824.00,20800000,5504997,Benign
3,17,100126,22,0,616,0,28,28,28.0,0.0000,...,32,0.000000e+00,0.000000e+00,0,0,0.0,0.00,0,0,Benign
4,0,54760,4,0,0,0,0,0,0.0,0.0000,...,0,0.000000e+00,0.000000e+00,0,0,0.0,0.00,0,0,Benign


In [24]:
print(master_df["Label"].value_counts())

Label
Benign                        1518487
DoS Hulk                       172846
DDoS                           128014
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
PortScan                         1956
Web Attack � Brute Force         1470
Bot                              1437
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [25]:
print(master_df.shape)

(1854979, 78)


In [26]:
#Create Working Copy
df = master_df.copy()

In [27]:
#Check Missing Values
print(df.isnull().sum())

df = df.dropna()

print(df.shape)

Protocol                    0
Flow Duration               0
Total Fwd Packets           0
Total Backward Packets      0
Fwd Packets Length Total    0
                           ..
Idle Mean                   0
Idle Std                    0
Idle Max                    0
Idle Min                    0
Label                       0
Length: 78, dtype: int64
(1854979, 78)


In [28]:
print(df.columns.tolist())

['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ra

In [29]:
label_mapping = {
    "Benign": "Benign",

    "Bot": "Botnet",

    "DDoS": "DDoS",

    "DoS Hulk": "DoS",
    "DoS GoldenEye": "DoS",
    "DoS Slowhttptest": "DoS",
    "DoS slowloris": "DoS",

    "FTP-Patator": "Brute Force",
    "SSH-Patator": "Brute Force",

    "Web Attack � Brute Force": "Web Attack",
    "Web Attack � Sql Injection": "Web Attack",
    "Web Attack � XSS": "Web Attack",

    "Infiltration": "Infiltration",

    "PortScan": "PortScan",

    "Heartbleed": "Heartbleed"
}

df["Label"] = df["Label"].map(label_mapping)

In [30]:
print(df["Label"].value_counts())

Label
Benign          1518487
DoS              193745
DDoS             128014
Brute Force        9150
Web Attack         2143
PortScan           1956
Botnet             1437
Infiltration         36
Heartbleed           11
Name: count, dtype: int64


In [31]:
#Feature Engineering

if "Size_Symmetry" not in df.columns:
    df["Size_Symmetry"] = (
        df["Avg Packet Size"] /
        (df["Bwd Packet Length Max"] + 1)
    )

df.head()

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Size_Symmetry
0,6,112740690,32,16,6448,1152,403,0,201.5,204.7242,...,3.594286e+02,1.199802e+01,380,343,16100000.0,498804.80,16400000,15400000,Benign,2.283961
1,6,112740560,32,16,6448,5056,403,0,201.5,204.7242,...,3.202857e+02,1.574499e+01,330,285,16100000.0,498793.66,16400000,15400000,Benign,0.782532
2,0,113757377,545,0,0,0,0,0,0.0,0.0000,...,9.361829e+06,7.324646e+06,18900000,19,12200000.0,6935824.00,20800000,5504997,Benign,0.000000
3,17,100126,22,0,616,0,28,28,28.0,0.0000,...,0.000000e+00,0.000000e+00,0,0,0.0,0.00,0,0,Benign,29.272728
4,0,54760,4,0,0,0,0,0,0.0,0.0000,...,0.000000e+00,0.000000e+00,0,0,0.0,0.00,0,0,Benign,0.000000


In [ ]:
df.columns = df.columns.str.strip()

selected_features = [
    col for col in selected_features if col in df.columns
]

X = df[selected_features]
y = df["Label"]

KeyError: "['Log_Flow_Duration', 'Log_Fwd_IAT_Mean', 'Interaction_Density'] not in index"

In [33]:
#Standardize Features
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Feature Scaling Completed")

NameError: name 'X' is not defined

In [34]:
print(df.shape)

(1854979, 79)


In [35]:
#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])

NameError: name 'X_scaled' is not defined

In [36]:
#Extract Only Attack Samples
attack_classes = [
    "Botnet",
    "Brute Force",
    "DDoS",
    "DoS",
    "Infiltration",
    "PortScan",
    "Web Attack"
]

train_mask = y_train.isin(attack_classes)

X_train_attack = X_train[train_mask]

y_train_attack = y_train[train_mask]

print("Attack Training Samples:", X_train_attack.shape[0])

print(y_train_attack.value_counts())

NameError: name 'y_train' is not defined

In [37]:
#Balance Classes using SMOTE
smote = SMOTE(
    random_state=42,
    k_neighbors=5
)

X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_attack,
    y_train_attack
)

print("Balanced Dataset")

print(y_train_balanced.value_counts())

NameError: name 'X_train_attack' is not defined

In [38]:
#Encode Attack Labels
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(
    y_train_balanced
)

print(label_encoder.classes_)

NameError: name 'y_train_balanced' is not defined

In [39]:
print(sorted(df["Label"].unique()))

['Benign', 'Botnet', 'Brute Force', 'DDoS', 'DoS', 'Heartbleed', 'Infiltration', 'PortScan', 'Web Attack']


In [50]:
#Train XGBoost

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    tree_method="hist"
)

xgb_model.fit(
    X_train_balanced,
    y_train_encoded
)

print("XGBoost Training Completed Successfully!")

AttributeError: 'LabelEncoder' object has no attribute 'classes_'

In [41]:
#Prepare Test Attack Data & Predict

# Keep only attack samples in test data

test_mask = y_test.isin(attack_classes)

X_test_attack = X_test[test_mask]

y_test_attack = y_test[test_mask]

# Encode labels

y_test_encoded = label_encoder.transform(y_test_attack)

# Predictions

y_pred = xgb_model.predict(X_test_attack)

predicted_labels = label_encoder.inverse_transform(y_pred)

print("Prediction Completed")

NameError: name 'y_test' is not defined

In [42]:
#Classification Report

print("="*60)
print("Classification Report")
print("="*60)

print(
    classification_report(
        y_test_attack,
        predicted_labels,
        digits=4
    )
)

Classification Report


NameError: name 'y_test_attack' is not defined

In [43]:
#Confusion Matrix Heatmap
cm = confusion_matrix(
    y_test_attack,
    predicted_labels,
    labels=label_encoder.classes_
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("XGBoost Multi-Class Confusion Matrix")

plt.tight_layout()

plt.show()

NameError: name 'y_test_attack' is not defined

In [44]:
#Feature Importance
importance = pd.DataFrame({

    "Feature": selected_features,

    "Importance": xgb_model.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

NameError: name 'xgb_model' is not defined

In [45]:
#Feature Importance Plot
plt.figure(figsize=(8,5))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance")

plt.tight_layout()

plt.show()

NameError: name 'importance' is not defined

<Figure size 800x500 with 0 Axes>

In [46]:
#Precision–Recall Curves
from sklearn.preprocessing import label_binarize

y_score = xgb_model.predict_proba(X_test_attack)

y_test_bin = label_binarize(
    y_test_encoded,
    classes=range(len(label_encoder.classes_))
)

plt.figure(figsize=(10,8))

for i in range(len(label_encoder.classes_)):

    precision, recall, _ = precision_recall_curve(
        y_test_bin[:, i],
        y_score[:, i]
    )

    ap = average_precision_score(
        y_test_bin[:, i],
        y_score[:, i]
    )

    plt.plot(
        recall,
        precision,
        label=f"{label_encoder.classes_[i]} (AP={ap:.3f})"
    )

plt.xlabel("Recall")

plt.ylabel("Precision")

plt.title("Precision-Recall Curves")

plt.legend()

plt.grid(True)

plt.show()

NameError: name 'xgb_model' is not defined

In [47]:
#Save Model
joblib.dump(
    xgb_model,
    "xgboost_classifier.pkl"
)

print("Model Saved Successfully")

NameError: name 'xgb_model' is not defined

In [48]:
#Save Label Encoder
joblib.dump(
    label_encoder,
    "label_encoder.pkl"
)

print("Label Encoder Saved")

Label Encoder Saved


In [49]:
#Save Scaler

#This will be useful during Notebook 3 (Integration).

joblib.dump(
    scaler,
    "xgboost_scaler.pkl"
)

print("Scaler Saved")

Scaler Saved
